In this tutorial, the square of the values in an array will be calculated with OpenCL.

In [ ]:
!pip install pyopencl

In [1]:
import numpy as np
import pyopencl as cl

OpenCL needs a 'context' and 'queue' to operate. This is a standard procedure and often called 'boilerplate'.

In [2]:
ctx = cl.create_some_context()
queue = cl.CommandQueue(ctx)

The objective is to calculate the square of each element in an array. For this purpose, let us create an array with values from zero to *n*, and an empty output array. These arrays will be used on the *host*.

In [3]:
n = 10
h_array_input = np.arange(n, dtype=np.int32)
h_array_output = np.zeros(n, dtype=np.int32)

Since the calculations will be performed on the *device*, the input and output arrays need to be defined on the *device* as well. For this, we can use a ```Buffer```.

In [4]:
d_array_input = cl.Buffer(ctx,
                          cl.mem_flags.READ_ONLY | cl.mem_flags.COPY_HOST_PTR,
                          hostbuf=h_array_input)
d_array_output = cl.Buffer(ctx,
                           cl.mem_flags.WRITE_ONLY,
                           h_array_output.nbytes)

The calculation that will be executed has to be specified as a 'kernel'. This is a text string with OpenCL code.

In [5]:
kernel = """
__kernel void square(
    __global const int* a,
    __global int* b)
{
    int id = get_global_id(0);
    b[id] = a[id] * a[id];
}
"""

The code in the kernel needs to be compiled before we can use it. The compiled code will be stored in a 'program'.

In [6]:
prg = cl.Program(ctx, kernel).build()

We are now ready to launch the kernel.

In [7]:
event_kernel = prg.square(queue, h_array_input.shape, None, d_array_input, d_array_output)
# Notar que no le estamos dando workgroup size

After executing the kernel, the output data is available on the device but not yet on the host.

In [8]:
print("Input array:")
print(h_array_input)
print("Squared values:")
print(h_array_output)

Input array:
[0 1 2 3 4 5 6 7 8 9]
Squared values:
[0 0 0 0 0 0 0 0 0 0]


Now the calculations have been performed on the device, we need to copy the output to the host.

In [9]:
event_copy = cl.enqueue_copy(queue, h_array_output, d_array_output)

We are finally ready to print the output.

In [10]:
print("Input array:")
print(h_array_input)
print("Squared values:")
print(h_array_output)

Input array:
[0 1 2 3 4 5 6 7 8 9]
Squared values:
[ 0  1  4  9 16 25 36 49 64 81]
